# Lesson 5 — New columns & transformations
**Goal:** derive the numbers a manager actually asks for: revenue, profit, margin, order size bands, weekday.

In [1]:
import pandas as pd, numpy as np
orders   = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
products = pd.read_csv('data/products.csv')
orders = orders.merge(products[['product_id', 'unit_cost']], on='product_id')  # bring in cost (lesson 7 explains merge)

## Arithmetic on columns

In [2]:
orders['gross']    = orders['quantity'] * orders['unit_price']
orders['discount'] = orders['gross'] * orders['discount_pct'] / 100
orders['revenue']  = orders['gross'] - orders['discount']
orders['profit']   = orders['revenue'] - orders['quantity'] * orders['unit_cost']
orders['margin_%'] = (orders['profit'] / orders['revenue'] * 100).round(1)
orders[['order_id', 'quantity', 'unit_price', 'discount_pct', 'revenue', 'profit', 'margin_%']].head()

,order_id,quantity,unit_price,discount_pct,revenue,profit,margin_%
0,O00001,3,25000,0,75000.0,33000.0,44.0
1,O00002,2,12000,0,24000.0,14000.0,58.3
2,O00003,1,25000,5,23750.0,9750.0,41.1
3,O00004,1,45000,0,45000.0,19000.0,42.2
4,O00005,5,12000,0,60000.0,35000.0,58.3


## Conditional columns with `np.where` and `np.select`

In [3]:
orders['is_discounted'] = np.where(orders['discount_pct'] > 0, 'Yes', 'No')
conditions = [orders['revenue'] >= 500_000, orders['revenue'] >= 100_000]
orders['order_size'] = np.select(conditions, ['Large', 'Medium'], default='Small')
orders['order_size'].value_counts()

order_size
Small     1558
Medium     721
Large      121
Name: count, dtype: int64

## Binning numbers with `pd.cut`

In [4]:
orders['revenue_band'] = pd.cut(orders['revenue'],
                               bins=[0, 50_000, 200_000, 1_000_000, np.inf],
                               labels=['< 50k', '50k–200k', '200k–1M', '1M+'])
orders['revenue_band'].value_counts().sort_index()

revenue_band
< 50k        712
50k–200k    1398
200k–1M      229
1M+           61
Name: count, dtype: int64

## `map` for lookups and `apply` for custom logic

In [5]:
region = {'Kariakoo': 'Dar', 'Mlimani City': 'Dar', 'Arusha CBD': 'North', 'Mwanza Rock City': 'Lake'}
orders['region'] = orders['branch'].map(region)
orders['weekday'] = orders['order_date'].dt.day_name()
orders[['branch', 'region', 'order_date', 'weekday']].head()

,branch,region,order_date,weekday
0,Mlimani City,Dar,2025-01-01,Wednesday
1,Kariakoo,Dar,2025-01-01,Wednesday
2,Mlimani City,Dar,2025-01-01,Wednesday
3,Kariakoo,Dar,2025-01-02,Thursday
4,Kariakoo,Dar,2025-01-02,Thursday


In [6]:
def loyalty_points(row):
    pts = row['revenue'] // 10_000            # 1 point per TSh 10,000
    return pts * 2 if row['channel'] == 'Online' else pts   # double points online
orders['points'] = orders.apply(loyalty_points, axis=1)
orders[['revenue', 'channel', 'points']].head()

,revenue,channel,points
0,75000.0,In-store,7.0
1,24000.0,In-store,2.0
2,23750.0,Online,4.0
3,45000.0,Online,8.0
4,60000.0,In-store,6.0


> Tip: `apply(axis=1)` is flexible but slow on big data — prefer vectorised maths or `np.where` when you can.

In [7]:
print(f"Total profit: TSh {orders['profit'].sum():,.0f}  |  average margin {orders['profit'].sum()/orders['revenue'].sum():.1%}")

Total profit: TSh 98,967,300  |  average margin 31.0%


## Key takeaways
- New column = expression on existing columns.
- `np.where` (2 choices), `np.select` (many), `pd.cut` (number ranges), `map` (lookups), `apply` (custom, slower).

**Exercise:** add `price_tier`: 'Premium' if unit_price ≥ 200,000 else 'Standard'. What share of revenue is Premium?